In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [44]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from sklearn.metrics import r2_score, mean_squared_error

from kids.catalog.training_catalog import load_and_prepare_training_catalog, merge_training_catalogs
from kids.training_data import TrainingData
from kids.photoz_regressors import MDNConfig, MixtureDensityNetwork

In [103]:
kids_x_desi_path = '../data/KiDS_DR5_WCScut_x_DESI_DR1_columns.csv'
kids_x_sdss_path = '../data/KiDS_DR5_WCScut_x_SDSS_DR17_columns.csv'
model_path = '../saved_models/photoz/mdn_photoz_2026-10-12T10-59-17Z.pt'

In [104]:
device = 'cuda'

In [105]:
model = MixtureDensityNetwork(MDNConfig())
model.load_weights(model_path, device)
model = model.to(device)

In [106]:
kids_x_desi = load_and_prepare_training_catalog(kids_x_desi_path)
kids_x_sdss = load_and_prepare_training_catalog(kids_x_sdss_path)
merged_catalogs = merge_training_catalogs(kids_x_desi, kids_x_sdss)

In [107]:
training_data = TrainingData(merged_catalogs)
X_test, y_test = training_data.get_test_photoz()
X_test_tensor, y_test_tensor = torch.tensor(X_test, dtype=torch.float32, device=device), torch.tensor(y_test.values, dtype=torch.float32, device=device)

In [108]:
model.eval()
with torch.no_grad():
    preds = model(X_test_tensor)

In [110]:
y_pred = preds.mean.cpu().numpy()

In [111]:
def redshift_error(y_true, y_pred):
    nominator = y_pred - y_true
    denominator = y_true + 1
    redshift_err = nominator / denominator 
    return redshift_err.mean(), redshift_err.std()

In [113]:
nll = -preds.log_prob(y_test_tensor).mean()
print('Mean NLL:', nll.item())
print('MSE:', mean_squared_error(y_test, y_pred))
print('R2 score:',r2_score(y_test, y_pred))
mean, std = redshift_error(y_test, y_pred)
print(f'Redshift error: {mean:.5f} +/ {std:.5f}')

Mean NLL: -0.6333913803100586
MSE: 0.09021630783282912
R2 score: 0.8330178315253666
Redshift error: -0.00482 +/ 0.11408
